# EDA

Pada tahapan ini akan dilakukan eksplorasi data untuk menentukan aturan preprocessing pada tahap selanjutnya

In [1]:
# import library
from pathlib import Path
import math
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# biar file .py di folder utama bisa di-import dari folder notebooks
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from notebook_config import configure_notebook, tickers, ticker_names, palette, blue
from pipeline_utils import show_table

In [ ]:
# path folder data dan gaya grafik
root_dir, output_dir = configure_notebook()
data_dir = root_dir / "data"
interim_dir = data_dir / "interim"
interim_dir.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 150
plt.rcParams["font.family"] = "DejaVu Sans"
# nama kolom yang ditampilkan di tabel
display_names = {"snapshot_date": "Tanggal", "Kode Saham": "Ticker", "Open Price": "Open", "Tertinggi": "High", "Terendah": "Low", "Penutupan": "Close"}
# kolom harga OHLC dan kolom kunci (saham dan tanggal)
price_columns = ["Open Price", "Tertinggi", "Terendah", "Penutupan"]
key_columns = ["Kode Saham", "snapshot_date"]

## Load Data

File Excel harian digabung menjadi satu tabel. Tanggal diambil dari nama file, lalu nama kolom dan kode saham dirapikan.

In [3]:
# menggabungkan semua file harian Ringkasan Saham IDX jadi satu tabel
files = sorted(data_dir.glob("Ringkasan Saham-*.xlsx"))
if not files:
    raise FileNotFoundError("File Ringkasan Saham belum tersedia di folder data")

frames = []
# tanggal diambil dari nama file, contohnya Ringkasan Saham-20250102.xlsx
for file in files:
    date_match = re.fullmatch(r"Ringkasan Saham-(\d{8})\.xlsx", file.name)
    if date_match is None:
        raise ValueError(f"Tanggal pada nama file belum sesuai: {file.name}")
    frame = pd.read_excel(file, dtype={"Kode Saham": "string", "Remarks": "string", "Tanggal Perdagangan Terakhir": "string"})
    # merapikan spasi di nama kolom dan kode saham
    frame.columns = frame.columns.astype(str).str.strip()
    frame["Kode Saham"] = frame["Kode Saham"].astype("string").str.strip().str.upper()
    frame["snapshot_date"] = pd.to_datetime(date_match.group(1), format="%Y%m%d")
    frame["sumber_file"] = file.name
    frames.append(frame)

# menyatukan semua file lalu mengurutkan per saham dan tanggal
raw_data = pd.concat(frames, ignore_index=True)
raw_data = raw_data.sort_values(key_columns, kind="stable").reset_index(drop=True)
source_row_count = sum(len(frame) for frame in frames)
# mengecek tidak ada baris yang hilang waktu digabung
assert len(raw_data) == source_row_count
show_table(pd.DataFrame({"Keterangan": ["File sumber", "Baris", "Kolom", "Saham"], "Nilai": [len(files), len(raw_data), raw_data.shape[1], raw_data["Kode Saham"].nunique()]}))

,Keterangan,Nilai
0,File sumber,"1,447"
1,Baris,"1,203,620"
2,Kolom,30
3,Saham,982


## Pemeriksaan Kelengkapan Data

Pada tahap ini dilakukan pengecekan tipe data, missing values, dan duplikat.

In [4]:
# mengecek tipe data tiap kolom dan melihat 5 baris pertama
column_summary = raw_data.dtypes.astype(str).rename("Tipe Data").to_frame().rename_axis("Kolom")
show_table(column_summary, max_rows=len(column_summary))
show_table(raw_data.head(), display_names=display_names)

# mengambil 11 saham penelitian aja
panel = raw_data.loc[raw_data["Kode Saham"].isin(tickers)].copy()
# menghitung jumlah saham per tanggal, harusnya selalu 11
coverage = panel.groupby("snapshot_date")["Kode Saham"].agg(Saham="nunique", Baris="size")
incomplete_dates = coverage.loc[(coverage["Saham"] != len(tickers)) | (coverage["Baris"] != len(tickers))]
show_table(coverage.head().rename_axis("Tanggal"))
show_table(incomplete_dates.rename_axis("Tanggal"))

,Tipe Data
Kolom,
No,int64
Kode Saham,string
Nama Perusahaan,object
Remarks,string
Sebelumnya,int64
Open Price,int64
Tanggal Perdagangan Terakhir,string
First Trade,int64
Tertinggi,int64


,No,Ticker,Nama Perusahaan,Remarks,Sebelumnya,Open,Tanggal Perdagangan Terakhir,First Trade,High,Low,Close,Selisih,Volume,Nilai,Frekuensi,Index Individual,Offer,Offer Volume,Bid,Bid Volume,Listed Shares,Tradeble Shares,Weight For Index,Foreign Sell,Foreign Buy,Non Regular Volume,Non Regular Value,Non Regular Frequency,Tanggal,sumber_file
0,1,AADI,Adaro Andalan Indonesia Tbk.,--U-1130000000A121------------,"5,550",0,05 Des 2024,0,"6,650","6,650","6,650","1,100","459,400","3,055,010,000","1,891",119.8,0,0,"6,650","433,438,600","7,786,891,760","7,786,891,760","778,689,200","3,000",0,"4,913,500","30,282,545,600",74,2024-12-05,Ringkasan Saham-20241205.xlsx
1,1,AADI,Adaro Andalan Indonesia Tbk.,--U-1130000000A121------------,"6,650",0,06 Des 2024,0,"7,975","7,975","7,975","1,325","446,600","3,561,635,000","1,652",143.7,0,0,"7,975","228,535,700","7,786,891,760","7,786,891,760","778,689,200","2,000","55,000","4,995,100","34,909,034,500",35,2024-12-06,Ringkasan Saham-20241206.xlsx
2,1,AADI,Adaro Andalan Indonesia Tbk.,--UO1130000000A121------------,"7,975","9,550",09 Des 2024,"9,550","9,550","9,550","9,550","1,575","43,658,100","416,934,855,000","14,582",172.1,0,0,"9,550","72,552,300","7,786,891,760","7,786,891,760","778,689,200","4,926,900","700,000","5,527,560,351","32,951,804,289,280",132,2024-12-09,Ringkasan Saham-20241209.xlsx
3,1,AADI,Adaro Andalan Indonesia Tbk.,--UO1130000000A121------------,"9,550","10,500",10 Des 2024,"10,475","11,375","9,600","10,275",725,"228,636,600","2,386,633,100,000","140,177",185.1,"10,275","176,500","10,250","310,200","7,786,891,760","7,786,891,760","778,689,200","13,376,200","4,198,100","145,358,698","870,437,021,101",278,2024-12-10,Ringkasan Saham-20241210.xlsx
4,1,AADI,Adaro Andalan Indonesia Tbk.,--UO1130000000A121------------,"10,275","10,275",11 Des 2024,"10,250","10,450","9,575","9,600",-675,"100,219,900","990,807,535,000","75,058",173,"9,600","99,100","9,575","418,700","7,786,891,760","7,786,891,760","778,689,200","15,285,800","2,991,900","146,283,657","872,251,406,799",452,2024-12-11,Ringkasan Saham-20241211.xlsx


,Saham,Baris
Tanggal,,
2020-01-02 00:00:00,11,11
2020-01-03 00:00:00,11,11
2020-01-06 00:00:00,11,11
2020-01-07 00:00:00,11,11
2020-01-08 00:00:00,11,11


,Saham,Baris
Tanggal,,


In [5]:
# mengecek data kosong di semua saham dan di 11 saham penelitian
missing_summary = pd.DataFrame({"Kosong seluruh sumber": raw_data.isna().sum(), "Kosong objek penelitian": panel.isna().sum()}).rename_axis("Kolom")
show_table(missing_summary, max_rows=len(missing_summary))

# mengecek baris dobel, saham dan tanggal yang sama muncul lebih dari sekali
duplicates = panel.loc[panel.duplicated(key_columns, keep=False)]
show_table(pd.DataFrame({"Keterangan": ["Tanggal tidak lengkap", "Baris duplikat objek penelitian", "Baris duplikat seluruh sumber"], "Nilai": [len(incomplete_dates), len(duplicates), int(raw_data.duplicated(key_columns, keep=False).sum())]}))
show_table(duplicates[key_columns + price_columns + ["Volume"]], display_names=display_names)

,Kosong seluruh sumber,Kosong objek penelitian
Kolom,,
No,0,0
Kode Saham,0,0
Nama Perusahaan,0,0
Remarks,0,0
Sebelumnya,0,0
Open Price,0,0
Tanggal Perdagangan Terakhir,0,0
First Trade,0,0
Tertinggi,0,0


,Keterangan,Nilai
0,Tanggal tidak lengkap,0
1,Baris duplikat objek penelitian,0
2,Baris duplikat seluruh sumber,0


,Ticker,Tanggal,Open,High,Low,Close,Volume


Sebelas saham penelitian memiliki 15.917 baris pada 1.447 tanggal. Setiap tanggal berisi sebelas saham, seluruh kolom terisi, dan tidak ada pasangan ticker dengan tanggal duplikat.

## Statistik Deskriptif

Tahapan ini bertujuan untuk melihat deskripsi statistik pada kolom numerik

In [6]:
# statistik deskriptif tiap kolom angka per saham
numeric_columns = panel.select_dtypes(include="number").columns.tolist()
statistics = panel.groupby("Kode Saham")[numeric_columns].describe()
for ticker in tickers:
    stock_statistics = statistics.loc[ticker].unstack().rename_axis("Kolom")
    stock_statistics.insert(0, "Ticker", ticker)
    show_table(stock_statistics, max_rows=len(stock_statistics))

,Ticker,count,mean,std,min,25%,50%,75%,max
Kolom,,,,,,,,,
No,BBCA,"1,447",83.2654,9.65018,67,74,83,95,97
Sebelumnya,BBCA,"1,447","15,488.7","10,405.7","7,000","8,450","9,225","28,437.5","36,725"
Open Price,BBCA,"1,447","13,272.4","10,455",0,"8,050","9,000","10,350","36,725"
First Trade,BBCA,"1,447","13,274.1","10,450.9",0,"8,050","9,000","10,350","36,725"
Tertinggi,BBCA,"1,447","15,661.1","10,539.1","7,100","8,537.5","9,300","28,762.5","36,900"
Terendah,BBCA,"1,447","15,331.6","10,284.6","7,000","8,375","9,150","27,925","36,225"
Penutupan,BBCA,"1,447","15,491.4","10,409.5","7,000","8,450","9,225","28,437.5","36,725"
Selisih,BBCA,"1,447",2.71251,340.724,"-2,175",-100,0,100,"3,900"
Volume,BBCA,"1,447",7.10699e+07,6.34469e+07,5.46e+06,2.64034e+07,6.14052e+07,9.1261e+07,7.56432e+08


,Ticker,count,mean,std,min,25%,50%,75%,max
Kolom,,,,,,,,,
No,BBNI,"1,447",88.2654,9.65018,72,79,88,100,102
Sebelumnya,BBNI,"1,447","6,234.28","1,849.81","3,160","4,670","5,600","7,925","10,425"
Open Price,BBNI,"1,447","5,898.36","2,465.36",0,"4,555","5,600","7,900","10,450"
First Trade,BBNI,"1,447","5,899.28","2,465.65",0,"4,555","5,600","7,925","10,475"
Tertinggi,BBNI,"1,447","6,317.98","1,859.83","3,410","4,740","5,650","7,987.5","10,500"
Terendah,BBNI,"1,447","6,156.34","1,841.1","2,970","4,610","5,500","7,825","10,350"
Penutupan,BBNI,"1,447","6,235.45","1,852.97","3,160","4,670","5,600","7,925","10,425"
Selisih,BBNI,"1,447",1.17139,126.096,-750,-60,0,70,580
Volume,BBNI,"1,447",4.54979e+07,3.15811e+07,7.2968e+06,2.38262e+07,3.70143e+07,5.64768e+07,2.46728e+08


,Ticker,count,mean,std,min,25%,50%,75%,max
Kolom,,,,,,,,,
No,BBRI,"1,447",89.2654,9.65018,73,80,89,101,103
Sebelumnya,BBRI,"1,447","4,387.15",734.835,"2,170","3,930","4,360","4,810","6,400"
Open Price,BBRI,"1,447","4,154.12","1,358.05",0,"3,940","4,360","4,810","6,350"
First Trade,BBRI,"1,447","4,155.57","1,358.64",0,"3,950","4,360","4,810","6,350"
Tertinggi,BBRI,"1,447","4,441.79",730.815,"2,270","3,990","4,400","4,850","6,450"
Terendah,BBRI,"1,447","4,339",735.355,"2,160","3,880","4,310","4,760","6,350"
Penutupan,BBRI,"1,447","4,386.7",735.034,"2,170","3,930","4,360","4,810","6,400"
Selisih,BBRI,"1,447",-0.442294,85.7907,-410,-50,0,50,500
Volume,BBRI,"1,447",1.86934e+08,1.14022e+08,2.76765e+07,1.09273e+08,1.55011e+08,2.26727e+08,1.0006e+09


,Ticker,count,mean,std,min,25%,50%,75%,max
Kolom,,,,,,,,,
No,BBTN,"1,447",93.1016,9.87656,75,84,93,105,107
Sebelumnya,BBTN,"1,447","1,397.07",265.332,745,"1,235","1,350","1,570","2,160"
Open Price,BBTN,"1,447","1,311.29",454.167,0,"1,225","1,340","1,565","2,170"
First Trade,BBTN,"1,447","1,311.96",454.218,0,"1,225","1,345","1,565","2,170"
Tertinggi,BBTN,"1,447","1,420.2",266.99,760,"1,255","1,370","1,595","2,180"
Terendah,BBTN,"1,447","1,376.69",261.332,720,"1,220","1,330","1,550","2,130"
Penutupan,BBTN,"1,447","1,396.45",264.712,745,"1,235","1,350","1,570","2,160"
Selisih,BBTN,"1,447",-0.618521,34.0249,-210,-15,-5,15,180
Volume,BBTN,"1,447",3.84837e+07,4.55908e+07,2.7882e+06,1.5512e+07,2.63626e+07,4.42115e+07,6.0274e+08


,Ticker,count,mean,std,min,25%,50%,75%,max
Kolom,,,,,,,,,
No,BDMN,"1,447",98.5639,10.3111,80,89,98,111,113
Sebelumnya,BDMN,"1,447","2,665.04",329.023,"1,675","2,440","2,600","2,830","4,070"
Open Price,BDMN,"1,447",401.202,911.885,0,0,0,0,"2,630"
First Trade,BDMN,"1,447",401.113,911.68,0,0,0,0,"2,630"
Tertinggi,BDMN,"1,447","2,703.6",334.59,"1,735","2,470","2,630","2,860","4,130"
Terendah,BDMN,"1,447","2,633.09",322.853,"1,620","2,410","2,580","2,800","3,960"
Penutupan,BDMN,"1,447","2,664.02",327.322,"1,675","2,440","2,600","2,830","4,070"
Selisih,BDMN,"1,447",-1.02281,51.4095,-320,-20,0,20,350
Volume,BDMN,"1,447",2.74481e+06,4.05221e+06,"112,100","909,500",1.6353e+06,3.12215e+06,8.73077e+07


...

,Ticker,count,mean,std,min,25%,50%,75%,max
Kolom,,,,,,,,,
No,BMRI,"1,447",129.162,13.5698,104,116,129,144,148
Sebelumnya,BMRI,"1,447","6,538.64","1,533.39","3,720","5,400","6,250","7,250","10,900"
Open Price,BMRI,"1,447","6,149.85","2,312.77",0,"5,200","6,250","7,250","10,875"
First Trade,BMRI,"1,447","6,151.38","2,312.84",0,"5,200","6,250","7,250","10,875"
Tertinggi,BMRI,"1,447","6,627.13","1,541.81","3,800","5,500","6,325","7,325","11,000"
Terendah,BMRI,"1,447","6,457.35","1,523.81","3,660","5,325","6,150","7,175","10,725"
Penutupan,BMRI,"1,447","6,540.5","1,536.77","3,720","5,400","6,250","7,250","10,900"
Selisih,BMRI,"1,447",1.86593,136.092,-700,-75,0,75,725
Volume,BMRI,"1,447",8.62656e+07,7.06564e+07,3.5332e+06,4.24693e+07,6.68848e+07,1.0418e+08,7.38455e+08


,Ticker,count,mean,std,min,25%,50%,75%,max
Kolom,,,,,,,,,
No,BNGA,"1,447",134.162,13.5698,109,121,134,149,153
Sebelumnya,BNGA,"1,447","1,314.08",408.132,550,985,"1,175","1,715","2,200"
Open Price,BNGA,"1,447",297.771,652.059,0,0,0,0,"1,830"
First Trade,BNGA,"1,447",297.944,652.438,0,0,0,0,"1,835"
Tertinggi,BNGA,"1,447","1,330.91",409.201,575,995,"1,185","1,730","2,220"
Terendah,BNGA,"1,447","1,300.72",406.193,515,970,"1,165","1,705","2,170"
Penutupan,BNGA,"1,447","1,314.65",408.22,550,985,"1,175","1,715","2,200"
Selisih,BNGA,"1,447",0.570145,21.9369,-125,-10,0,10,160
Volume,BNGA,"1,447",7.60313e+06,9.77042e+06,"467,200",3.04355e+06,5.2079e+06,8.654e+06,1.51156e+08


,Ticker,count,mean,std,min,25%,50%,75%,max
Kolom,,,,,,,,,
No,BNLI,"1,447",136.162,13.5698,111,123,136,151,155
Sebelumnya,BNLI,"1,447","1,657.8","1,001.05",870,997.5,"1,240","2,050","5,975"
Open Price,BNLI,"1,447",466.9,"1,226.78",0,0,0,0,"6,000"
First Trade,BNLI,"1,447",466.99,"1,227.44",0,0,0,0,"6,000"
Tertinggi,BNLI,"1,447","1,689.7","1,031.73",0,"1,010","1,255","2,090","6,500"
Terendah,BNLI,"1,447","1,624.26",963.218,0,985,"1,220","2,000","5,625"
Penutupan,BNLI,"1,447","1,660.75","1,005.52",870,997.5,"1,240","2,050","5,975"
Selisih,BNLI,"1,447",2.95439,70.6336,-480,-10,0,10,650
Volume,BNLI,"1,447",2.81098e+06,8.45488e+06,0,"68,150","180,100","928,250",1.44161e+08


,Ticker,count,mean,std,min,25%,50%,75%,max
Kolom,,,,,,,,,
No,BTPS,"1,447",159.764,15.6335,133,144,160,176,182
Sebelumnya,BTPS,"1,447","2,420.77","1,036.81",830,"1,380","2,450","3,260","5,050"
Open Price,BTPS,"1,447",884.091,"1,411.79",0,0,0,"1,360","5,050"
First Trade,BTPS,"1,447",884.672,"1,412.65",0,0,0,"1,360","5,025"
Tertinggi,BTPS,"1,447","2,472.8","1,059.89",850,"1,412.5","2,520","3,350","5,125"
Terendah,BTPS,"1,447","2,375.63","1,015.63",800,"1,365","2,380","3,200","4,880"
Penutupan,BTPS,"1,447","2,418.67","1,036.19",830,"1,377.5","2,450","3,260","5,050"
Selisih,BTPS,"1,447",-2.10435,79.8601,-310,-35,-5,30,550
Volume,BTPS,"1,447",1.09121e+07,1.20873e+07,"281,600",4.50035e+06,7.4228e+06,1.24134e+07,1.6222e+08


,Ticker,count,mean,std,min,25%,50%,75%,max
Kolom,,,,,,,,,
No,PNBN,"1,447",598.287,70.8071,488,527,594,670,701
Sebelumnya,PNBN,"1,447","1,200.95",394.292,645,840,"1,130","1,365","2,780"
Open Price,PNBN,"1,447",220.377,506.591,0,0,0,0,"2,020"
First Trade,PNBN,"1,447",220.404,506.69,0,0,0,0,"2,030"
Tertinggi,PNBN,"1,447","1,227.2",408.104,690,860,"1,150","1,400","2,790"
Terendah,PNBN,"1,447","1,177.7",380.898,615,825,"1,115","1,340","2,670"
Penutupan,PNBN,"1,447","1,200.77",394.289,645,840,"1,130","1,365","2,780"
Selisih,PNBN,"1,447",-0.176227,40.2136,-220,-15,0,10,350
Volume,PNBN,"1,447",9.2055e+06,1.41007e+07,"167,500",2.50925e+06,5.0755e+06,1.04905e+07,1.99682e+08


## Pemeriksaan Harga dan Volume

Dikarenakan ditemukan nilai minimum nol pada tahap statistik deskriptif, tahapan ini akan dilakukan pemeriksaan baris tersebut.

In [7]:
# close hari sebelumnya di saham yang sama
# mencari baris yang aneh, misalnya harga atau volume nol
previous_close = panel.groupby("Kode Saham")["Penutupan"].shift(1)
flat_price = panel[price_columns].eq(panel["Penutupan"], axis=0).all(axis=1)
zero_volume = panel["Volume"].eq(0)
zero_ohl = panel[["Open Price", "Tertinggi", "Terendah"]].eq(0).all(axis=1)
unchanged_close = panel["Penutupan"].eq(previous_close)
# pola tidak ada transaksi: volume nol, harga datar atau OHL nol, close tidak berubah
non_trading = (zero_volume & (flat_price | (zero_ohl & unchanged_close))) | (flat_price & unchanged_close)

# menghitung jumlah nilai nol per kolom di tiap saham
zero_counts = panel[price_columns + ["Volume"]].eq(0).groupby(panel["Kode Saham"]).sum().reindex(tickers)
zero_counts["Pola tanpa perdagangan"] = non_trading.groupby(panel["Kode Saham"]).sum().reindex(tickers)
show_table(zero_counts, display_names=display_names)

# menampilkan contoh baris bermasalah bareng close hari sebelumnya
problem_mask = panel[price_columns + ["Volume"]].le(0).any(axis=1) | non_trading
problem_rows = panel.loc[problem_mask, key_columns + price_columns + ["Volume"]].copy()
problem_rows["Close Sebelumnya"] = previous_close.loc[problem_mask]
show_table(problem_rows.head(15), display_names=display_names)
show_table(panel.loc[zero_volume | panel["Tertinggi"].eq(0) | panel["Terendah"].eq(0), key_columns + price_columns + ["Volume"]], display_names=display_names)

,Open,High,Low,Close,Volume,Pola tanpa perdagangan
Ticker,,,,,,
BBCA,114,0,0,0,0,0
BBNI,114,0,0,0,0,0
BBRI,114,0,0,0,0,0
BBTN,114,0,0,0,0,0
BDMN,"1,212",0,0,0,0,0
BJBR,"1,197",0,0,0,0,0
BMRI,114,0,0,0,0,0
BNGA,"1,197",0,0,0,0,0
BNLI,"1,226",1,1,0,1,1


,Ticker,Tanggal,Open,High,Low,Close,Volume,Close Sebelumnya
119089,BBCA,2020-03-13,0,"28,750","25,875","28,300","42,338,000","27,800"
119090,BBCA,2020-03-16,0,"28,300","26,575","27,525","30,109,700","28,300"
119091,BBCA,2020-03-17,0,"27,525","25,600","25,600","42,440,000","27,525"
119092,BBCA,2020-03-18,0,"26,900","23,825","25,000","36,010,700","25,600"
119093,BBCA,2020-03-19,0,"24,975","23,250","23,250","52,883,800","25,000"
119094,BBCA,2020-03-20,0,"26,925","21,625","23,675","67,026,400","23,250"
119095,BBCA,2020-03-23,0,"25,000","22,025","22,150","43,568,700","23,675"
119096,BBCA,2020-03-24,0,"24,975","22,250","22,500","50,794,200","22,150"
119097,BBCA,2020-03-26,0,"26,975","23,250","26,400","66,326,600","22,500"
119098,BBCA,2020-03-27,0,"29,475","26,775","27,550","53,936,600","26,400"


,Ticker,Tanggal,Open,High,Low,Close,Volume
196963,BNLI,2025-09-30,0,0,0,"5,950",0


BNLI pada 2025-09-30 memiliki Open, High, Low, dan Volume bernilai nol, sedangkan Close sama dengan hari sebelumnya. Setelah investigasi, pada hari tersebut BNLI mengalami suspensi sehingga tidak terdapat perdangan. Maka dari itu pada tahap preprocessing, keempat harga OHLC pada baris ini akan diisi menggunakan Close hari sebelumnya dan Volume tetap nol. Nilai Open nol pada baris lainnya dipertahankan karena indikator yang digunakan tidak memakai harga Open.

## Perubahan Jumlah Saham

Perubahan Jumlah Saham (Listed Shares) diperiksa pada setiap saham dan tanggal.

In [8]:
# mencari tanggal waktu jumlah saham beredar (Listed Shares) berubah
share_changes = panel[key_columns + ["Listed Shares"]].copy().sort_values(key_columns)
previous_shares = share_changes.groupby("Kode Saham")["Listed Shares"].shift()
share_changes["Selisih saham"] = share_changes["Listed Shares"] - previous_shares
# kalau rasionya bulat, misalnya 5, berarti stock split. ini dipakai di notebook 02
share_changes["Rasio saham"] = share_changes["Listed Shares"] / previous_shares
# menampilkan baris yang Listed Shares-nya berubah aja
share_changes = share_changes.loc[previous_shares.notna() & share_changes["Selisih saham"].ne(0)].copy()
show_table(share_changes, display_names=display_names, max_rows=len(share_changes))

,Ticker,Tanggal,Listed Shares,Selisih saham,Rasio saham
119471,BBCA,2021-10-13,"122,042,299,500",9.76338e+10,5
127188,BBNI,2023-10-06,"36,924,339,786",1.84622e+10,2
128133,BBRI,2021-09-15,"138,494,356,303",1.6382e+10,1.13416
128135,BBRI,2021-09-17,"139,020,826,472",5.2647e+08,1.0038
128136,BBRI,2021-09-20,"139,248,137,400",2.27311e+08,1.00164
128137,BBRI,2021-09-21,"139,477,938,957",2.29802e+08,1.00165
128138,BBRI,2021-09-22,"142,992,370,234",3.51443e+09,1.0252
128139,BBRI,2021-09-23,"145,900,886,219",2.90852e+09,1.02034
128140,BBRI,2021-09-24,"149,593,129,867",3.69224e+09,1.02531
128142,BBRI,2021-09-28,"150,325,543,504",7.32414e+08,1.0049


Perubahan Listed Shares ditampilkan sebagai temuan pada data ini. Pada tahap preprocessing, rasio yang menunjukkan stock split digunakan untuk menyesuaikan harga.

## Simpan Data

Hasil penggabungan data disimpan ke panel_raw.parquet untuk digunakan pada tahap preprocessing.

In [9]:
# menyimpan data gabungan buat dipakai di notebook 02
raw_path = interim_dir / "panel_raw.parquet"
raw_data.to_parquet(raw_path, index=False)
assert len(raw_data) == source_row_count
show_table(pd.DataFrame({"File": [str(raw_path.relative_to(root_dir))], "Baris": [len(raw_data)], "Kolom": [raw_data.shape[1]]}))

,File,Baris,Kolom
0,data\interim\panel_raw.parquet,"1,203,620",30
